# Behavioral Churn Prediction in E-Commerce Systems
## Olist Brazilian E-Commerce Dataset

**Author**: Nikhil Anita Sunil Wankhede  
**Date**: 2026-05-07  
**Dataset**: Brazilian E-Commerce Public Dataset by Olist (2016-2018)  

---

### Executive Summary

This notebook orchestrates a complete, leakage-free churn prediction pipeline
built on rigorous temporal methodology.  Churn is defined as **180 days of
inactivity** after a customer's last purchase.

Three classifiers are trained and evaluated; the best model receives SHAP
interpretability, statistical testing, customer segmentation, ablation study,
and production-grade risk scoring.

### Repository Structure

All heavy logic resides in the `src/` modules.  This notebook only invokes
the master pipeline and provides commentary.

### Methodology

- **Churn window**: 180 days of inactivity
- **Temporal split**: 70th percentile train cutoff; test cutoff = max date - 180 days
- **Features**: recency, frequency, monetary, delivery, review, payment
- **Models**: Logistic Regression, Random Forest, XGBoost (with early stopping)
- **Evaluation**: ROC-AUC, PR-AUC, F1, calibration, threshold analysis
- **Explainability**: SHAP summary/bar/dependence plots
- **Segmentation**: K-Means with PCA visualisation
- **Statistical rigour**: Mann-Whitney U tests with BH correction, Cliff's delta


In [ ]:
# ── Install dependencies (Kaggle only) ──
import sys, os, subprocess, importlib

required = ['xgboost', 'shap', 'pingouin', 'statsmodels', 'openpyxl']
missing = [pkg for pkg in required if importlib.util.find_spec(pkg) is None]
if missing:
    print(f"Installing missing packages: {missing}")
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])
else:
    print("All required packages already installed.")

In [ ]:
import sys
import os

# Add project root to path
project_root = os.path.dirname(os.path.dirname(os.path.abspath('__file__'))) if '__file__' in dir() else (
    '/kaggle/working' if os.path.exists('/kaggle/working') else os.getcwd()
)

if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f"Project root: {project_root}")
print(f"Python: {sys.version}")

In [ ]:
from src.pipeline import main
from src.config import ON_KAGGLE

print(f"Kaggle environment: {ON_KAGGLE}")

In [ ]:
%%time
main()

## Results & Interpretation

After running, all outputs are saved to `figures/` and `results/`.

Key artefacts:

- **Churn rate**: see `results/model_metrics/model_metrics.csv`
- **Model comparison**: ROC curves, PR curves, confusion matrices
- **Feature importance**: SHAP bar and summary plots
- **Calibration**: calibration curves with bootstrap CI
- **Customer segments**: PCA-projected K-Means clusters
- **Statistical tests**: Mann-Whitney with BH correction
- **Risk scores**: per-customer churn probability with tier assignment

### Behavioural Insights

Churners typically exhibit:
- Longer recency (days since last purchase)
- Lower order frequency
- Lower total spend
- Poorer review scores

### Limitations

- Data covers only 2016-2018
- 180-day churn window is fixed
- No causal inference (SHAP provides correlation)
- Static model — no survival / temporal dynamics

In [ ]:
print("Pipeline complete.  Outputs:")
for d in ['figures', 'results', 'models', 'processed_data']:
    path = os.path.join(project_root, d)
    if os.path.isdir(path):
        items = os.listdir(path)
        print(f"  {d}/  ({len(items)} items)")